We will analyze the Credit Card fraud dataset

In [1]:
!pip install pyod

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.2/58.2 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.4/411.4 kB 12.3 MB/s eta 0:00:00


In [9]:
import pandas as pd
import numpy as np
import kagglehub
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, average_precision_score
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from pyod.models.hbos import HBOS
import lightgbm as lgb
from sklearn.model_selection import train_test_split

In [3]:
path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")

print("Path to dataset files:", path)

100%|██████████| 66.0M/66.0M [00:01<00:00, 59.8MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/mlg-ulb/creditcardfraud/versions/3


In [4]:
df = pd.read_csv(path + '/creditcard.csv')

In [ ]:
print('First 5 rows:')
print(df.head())
print('Credit card info:')
print(df.info())
print('Credit card summary stats:')
print(df.describe())

First 5 rows:
   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
3  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575  0.647376   
4 -0.270533  0.817739  ... -0.009431  0.798278 -0.137458  0.141267 -0.206010   

        V26       

In [5]:
fraud_percentage = df['Class'].value_counts()[1] / len(df)
print(f"Fraud percentage: {100 * fraud_percentage:.4f}%")

Fraud percentage: 0.1727%


Isolation Forest

In [6]:
X = df.drop('Class', axis=1)
y = df['Class']

In [7]:
model_if = IsolationForest(contamination=fraud_percentage, random_state=42)
model_if.fit(X)

IsolationForest(contamination=np.float64(0.001727485630620034), random_state=42)

In [10]:
y_pred_if = model_if.predict(X)
y_pred_if_mapped = np.where(y_pred_if == -1, 1, 0)

pr_auc_if = average_precision_score(y, y_pred_if_mapped)

print('Isolation Forest Results:')
print(f'PR-AUC: {pr_auc_if:.4f}')
print(classification_report(y, y_pred_if_mapped))

Isolation Forest Results:
PR-AUC: 0.0669
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    284315
           1       0.26      0.26      0.26       492

    accuracy                           1.00    284807
   macro avg       0.63      0.63      0.63    284807
weighted avg       1.00      1.00      1.00    284807



Histogram-Based Outlier Score (HBOS)

In [11]:
model_hbos = HBOS(contamination=fraud_percentage)
model_hbos.fit(X)

HBOS(alpha=0.1, contamination=np.float64(0.001727485630620034), n_bins=10,
   tol=0.5)

In [12]:
y_pred_hbos = model_hbos.predict(X)

pr_auc_hbos = average_precision_score(y, y_pred_hbos)

print('HBOS Results:')
print(f'PR-AUC: {pr_auc_hbos:.4f}')
print(classification_report(y, y_pred_hbos))

HBOS Results:
PR-AUC: 0.1379
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    284315
           1       0.37      0.37      0.37       492

    accuracy                           1.00    284807
   macro avg       0.68      0.68      0.68    284807
weighted avg       1.00      1.00      1.00    284807



Gaussian Mixture Models (GMMs)

In [14]:
X_scaled = StandardScaler().fit_transform(X)

In [15]:
model_gmm = GaussianMixture(n_components=2, random_state=42)
model_gmm.fit(X_scaled)

GaussianMixture(n_components=2, random_state=42)

In [16]:
anomaly_scores_gmm = model_gmm.score_samples(X_scaled)
threshold = np.percentile(anomaly_scores_gmm, fraud_percentage * 100)
y_pred_gmm = (anomaly_scores_gmm < threshold).astype(int)

In [17]:
pr_auc_gmm = average_precision_score(y, y_pred_gmm)

print('Gaussian Mixture Models Results:')
print(f'PR-AUC: {pr_auc_gmm:.4f}')
print(classification_report(y, y_pred_gmm))

Gaussian Mixture Models Results:
PR-AUC: 0.0550
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    284315
           1       0.23      0.23      0.23       492

    accuracy                           1.00    284807
   macro avg       0.62      0.62      0.62    284807
weighted avg       1.00      1.00      1.00    284807



LightGBM classifier

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [ ]:
model_lgbm = lgb.LGBMClassifier(random_state=42, class_weight='balanced', n_jobs=8)
model_lgbm.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 344, number of negative: 199020
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.062786 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7650
[LightGBM] [Info] Number of data points in the train set: 199364, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.001725 -> initscore=-6.360519
[LightGBM] [Info] Start training from score -6.360519
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gai

LGBMClassifier(is_unbalance=True, random_state=42)

In [ ]:
y_pred_lgbm = model_lgbm.predict(X_test)

pr_auc_lgbm = average_precision_score(y_test, y_pred_lgbm)

print('LightGBM Classifier Results:')
print(f'PR-AUC: {pr_auc_lgbm:.4f}')
print(classification_report(y_test, y_pred_lgbm))

I had to run lightGBM locally.
```
LightGBM Classifier Results:
PR-AUC: 0.6937
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     85295
           1       0.86      0.80      0.83       148

    accuracy                           1.00     85443
   macro avg       0.93      0.90      0.92     85443
weighted avg       1.00      1.00      1.00     85443
```



Random Forest classifier

In [ ]:

model_rf = RandomForestClassifier(random_state=42, class_weight='balanced', n_jobs=8)
model_rf.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', random_state=42)

In [ ]:
y_pred_rf = model_rf.predict(X_test)

pr_auc_rf = average_precision_score(y_test, y_pred_rf)

print('Random Forest Classifier Results:')
print(f'PR-AUC: {pr_auc_rf:.4f}')
print(classification_report(y_test, y_pred_rf))

I had to run RF locally:

```
Random Forest Classifier Results:
PR-AUC: 0.6835
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     85295
           1       0.97      0.70      0.82       148

    accuracy                           1.00     85443
   macro avg       0.99      0.85      0.91     85443
weighted avg       1.00      1.00      1.00     85443
```



### Model Comparison

Overall, as expected, supervised models like lightGBM and Random Forest performed best with PR-AUC 0.69 and 0.68, respectively. Assuming that customer behavior cannot be predicted in advance, and can be changed during time, unsupervised techniques are approperiate with PR-AUC of 0.07, 0.14, 0.06 for Isolation Forest, HBOS, and GMM, respectively.